# Do scheduled token unlocks help forecast crypto volatility?

**MH6805 group project, Group 7.** One notebook, the whole analysis pipeline. Data gathering (Tokenomist API + Binance klines) is outside the scope of this notebook; it starts from `data/dataset_v1.csv`, whose columns are documented in `data/COLUMN_GUIDE.md`.

## Question
Do scheduled token-unlock (emission) features add forecasting power for a token's **forward 14-day realized volatility** beyond volatility persistence and market regime?

Every model is run twice: on **control features only** (trailing volatility, trailing returns, BTC/ETH volatility, size, age) and on **controls + emission + static** features. The difference is the finding. A null or negative result is a valid result; nothing is tuned until it disappears.

## Data
- Panel of 176 tokens x calendar day, 2024-09-14 to 2026-08-15, one row per token-day.
- Target `y = log(rv_fwd_14d)`: annualized realized volatility of daily log returns over the next 14 days.
- Emission features come from the unlock schedule as published, so they are known in advance (the only forward-looking inputs allowed). Caveat: schedules are the version curated on the pull date, so they are lookahead-safe mechanically but not against later revisions.

## Protocol (never shuffled)
- 5 walk-forward folds, tokens pooled, cutoffs every 90 days over the last 450 days.
- train = date <= cutoff - 30 days (embargo, because the target looks ahead); test = the 90 days after the cutoff.
- The last 90 days of train are the validation slice; tuning fits on train minus validation minus a 30-day purge, picks the config with the lowest validation RMSE, refits on all of train and scores **once** on test. Tuning never touches test.
- Metrics in log-vol space: RMSE, MAE, Spearman, and `r2_oos = 1 - SSE_model / SSE_persistence` where persistence = `log(rv_past_14d)`. `r2_oos = 0` means the model ties the persistence forecast.
- Scalers and models are fit on the training slice of the current fold only (they live inside each sklearn `Pipeline`). `random_state = 42` everywhere.

## How to run
Run from the repository root with `data/dataset_v1.csv` in place (`pip install -r requirements.txt`, Python 3.13). Cells are sequential.

Model runs are expensive (the Random Forest section is ~700 fits of 400 trees, several hours on 8 cores), so each run is stored in `results/<model>.csv` (one row per fold) and **reused on the next execution**; the notebook then only refits the two interpretation models and rebuilds every table and figure (about 20 minutes). Set the environment variable `NB_FRESH=1` before starting the kernel to recompute every run from scratch, or `NB_SMOKE=1` to run tiny models as a code check (nothing is saved in that mode).

Outputs are written to `results/` (one CSV per model, one row per fold per run; test-set predictions as parquet) and `figures/` (PNG copies of every chart shown below).


## 0. Shared harness

Data cleaning, feature sets, walk-forward folds, metrics, baselines and the generic tune / refit / predict loop. This is the content of `common.py` in the repository, inlined so the notebook is self-contained.

Feature sets (names after cleaning):
- `CONTROLS` (9): `rv_past_7d/14d/30d`, `ret_past_7d/30d`, `btc_rv_past_14d`, `eth_rv_past_14d`, `log_mcap_pit`, `days_since_tge`
- `EMISSION` (13): `emis_next_{7,30,90}d_pct_float` (log1p), `emis_next_30d_pct_total`, `days_to_next_cliff` (null -> 365), `next_cliff_pct_float` (log1p, null -> 0), `has_unlock_events`, six `emis_next_30d_<recipient>_pct_float` (log1p)
- `STATIC` (10): six `alloc_share_<recipient>`, `listedMethod` one-hots, `unlockPrecision_ord`
- `FULL` = CONTROLS + EMISSION + STATIC (32); `CONTROLS+EMIS5` adds the five headline emission columns only.


In [ ]:
# ---- imports ----------------------------------------------------------------
import json
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import ElasticNet, LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             log_loss, roc_auc_score)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# NB_SMOKE=1 runs tiny models and saves nothing (code check only).
SMOKE = os.environ.get("NB_SMOKE") == "1"
print("python", sys.version.split()[0], "| SMOKE =", SMOKE)

# ---- shared harness (identical to common.py in the repository) ----------------
import itertools
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RANDOM_STATE = 42

ROOT = Path.cwd()   # run from the repo root
DATA_PATH = ROOT / "data" / "dataset_v1.csv"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"

# --------------------------------------------------------------------------
# Column bookkeeping
# --------------------------------------------------------------------------
BUCKETS = ["founderTeam", "privateInvestors", "community",
           "publicInvestors", "reserve", "others"]

RAW_TARGETS = ["rv_fwd_7d", "rv_fwd_14d", "rv_fwd_30d",
               "rv_fwd_14d_parkinson", "fwd_ret_14d"]

# cleaned target column -> tag written to the `target` column of results CSVs
TARGETS = {
    "y": "log_rv_fwd_14d",
    "y_7d": "log_rv_fwd_7d",
    "y_30d": "log_rv_fwd_30d",
    "y_14d_parkinson": "log_rv_fwd_14d_parkinson",
}
_TARGET_SOURCE = {
    "y": "rv_fwd_14d",
    "y_7d": "rv_fwd_7d",
    "y_30d": "rv_fwd_30d",
    "y_14d_parkinson": "rv_fwd_14d_parkinson",
}

# persistence benchmark used in r2_oos, for every target
PERSIST_COL = "rv_past_14d"

LISTED_METHODS = ["INTERNAL", "AI", "EXTERNAL"]
PRECISION_ORDER = ["second", "hour", "block", "day", "week", "month", "unknown"]

RAW_EMIS_COUNT_COLS = (["emis_next_7d", "emis_next_30d", "emis_next_90d"]
                       + [f"emis_next_30d_{b}" for b in BUCKETS]
                       + ["emis_next_30d_unattributed"])
DROP_COLS = (["symbol", "tge_proxy_date", "total_supply_basis", "total_supply",
              "float_pit", "listedMethod", "unlockPrecision"]
             + RAW_EMIS_COUNT_COLS + RAW_TARGETS)

# --------------------------------------------------------------------------
# Feature sets (names are the column names AFTER load_data cleaning)
# --------------------------------------------------------------------------
CONTROLS = ["rv_past_7d", "rv_past_14d", "rv_past_30d", "ret_past_7d",
            "ret_past_30d", "btc_rv_past_14d", "eth_rv_past_14d",
            "log_mcap_pit", "days_since_tge"]

EMISSION_BUCKETS = [f"emis_next_30d_{b}_pct_float" for b in BUCKETS]
EMISSION = (["emis_next_7d_pct_float", "emis_next_30d_pct_float",
             "emis_next_90d_pct_float", "emis_next_30d_pct_total",
             "days_to_next_cliff", "next_cliff_pct_float", "has_unlock_events"]
            + EMISSION_BUCKETS)

STATIC = ([f"alloc_share_{b}" for b in BUCKETS]
          + [f"listedMethod_{m}" for m in LISTED_METHODS]
          + ["unlockPrecision_ord"])

FULL = CONTROLS + EMISSION + STATIC

EMIS5 = ["emis_next_30d_pct_float", "emis_next_7d_pct_float",
         "next_cliff_pct_float", "days_to_next_cliff", "has_unlock_events"]

FEATURE_SETS: dict[str, list[str]] = {
    "CONTROLS": CONTROLS,
    "EMISSION": EMISSION,
    "STATIC": STATIC,
    "FULL": FULL,
    "CONTROLS+EMIS5": CONTROLS + EMIS5,
    # recipient-split variant: the six bucket columns carry the 30d signal
    "FULL_NO_AGG30": [c for c in FULL if c != "emis_next_30d_pct_float"],
}


def register_feature_set(name: str, columns: list[str]) -> None:
    """Add a named feature set so run_model can look it up."""
    missing = [c for c in columns if c not in _cleaned_columns()]
    if missing:
        raise KeyError(f"feature set {name!r}: unknown columns {missing}")
    FEATURE_SETS[name] = list(columns)


def _cleaned_columns(path: Path = DATA_PATH) -> set[str]:
    """Column names load_data will produce, derived from the CSV header only."""
    raw = list(pd.read_csv(path, nrows=0).columns)
    cols = [c for c in raw if c not in DROP_COLS]
    cols += list(TARGETS)
    cols += EMISSION_BUCKETS
    cols += [f"listedMethod_{m}" for m in LISTED_METHODS]
    cols += ["unlockPrecision_ord"]
    return set(cols)


def _verify_feature_sets() -> None:
    have = _cleaned_columns()
    for name, cols in FEATURE_SETS.items():
        missing = [c for c in cols if c not in have]
        if missing:
            raise KeyError(f"feature set {name}: columns missing from "
                           f"{DATA_PATH.name} after cleaning: {missing}")
        dupes = {c for c in cols if cols.count(c) > 1}
        if dupes:
            raise ValueError(f"feature set {name}: duplicated columns {dupes}")
        leaked = [c for c in cols if c in RAW_TARGETS or c in TARGETS]
        if leaked:
            raise ValueError(f"feature set {name} contains targets: {leaked}")


_verify_feature_sets()


# --------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------
def load_data(path=DATA_PATH, freq: str = "daily", verbose: bool = True) -> pd.DataFrame:
    """Load and clean the panel. Returns token_id, date, targets and features,
    sorted by (token_id, date) with a fresh 0..n-1 index. Never shuffled."""
    if freq not in ("daily", "weekly"):
        raise ValueError("freq must be 'daily' or 'weekly'")

    def log(step, before, after):
        if verbose:
            print(f"[load_data] {step:<44s} rows {before:>7,d} -> {after:>7,d}")

    df = pd.read_csv(path)
    n0 = len(df)

    # 1. parse date, sort
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["token_id", "date"], kind="stable").reset_index(drop=True)
    log("1 parse date, sort (token_id, date)", n0, len(df))

    # 2. drop rows without target / 30d history
    n = len(df)
    df = df[df["rv_fwd_14d"].notna() & df["rv_past_30d"].notna()].copy()
    log("2 drop null rv_fwd_14d / rv_past_30d", n, len(df))

    # 3. log targets
    for new, src in _TARGET_SOURCE.items():
        df[new] = np.log(df[src])

    # 4. log1p the long-tailed pct_float columns (nulls stay null until step 5)
    pct_float_cols = [c for c in df.columns
                      if c.startswith("emis_next_") and c.endswith("_pct_float")]
    for c in pct_float_cols + ["next_cliff_pct_float"]:
        df[c] = np.log1p(df[c])

    # 5. structural nulls (tokens without cliffs / no cliff left in window)
    df["days_to_next_cliff"] = df["days_to_next_cliff"].fillna(365)
    df["next_cliff_pct_float"] = df["next_cliff_pct_float"].fillna(0.0)

    # 6. recipient buckets as share of float, log1p
    for b in BUCKETS:
        df[f"emis_next_30d_{b}_pct_float"] = np.log1p(
            df[f"emis_next_30d_{b}"] / df["float_pit"])

    # 7. encodings with fixed category lists (nothing is learned from data)
    unknown_lm = set(df["listedMethod"].dropna().unique()) - set(LISTED_METHODS)
    if unknown_lm:
        raise ValueError(f"unexpected listedMethod values: {unknown_lm}")
    for m in LISTED_METHODS:
        df[f"listedMethod_{m}"] = (df["listedMethod"] == m).astype(int)
    prec = df["unlockPrecision"].fillna("unknown")
    unknown_p = set(prec.unique()) - set(PRECISION_ORDER)
    if unknown_p:
        raise ValueError(f"unexpected unlockPrecision values: {unknown_p}")
    df["unlockPrecision_ord"] = prec.map(
        {p: i for i, p in enumerate(PRECISION_ORDER)}).astype(int)
    for c in ["has_unlock_events", "emission_schedule_ended"]:
        df[c] = df[c].astype(int)

    # 8. drop identifiers, raw token counts and every raw target
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # 9. weekly: first available day of each ISO week per token
    if freq == "weekly":
        n = len(df)
        iso = df["date"].dt.isocalendar()
        df = (df.assign(_y=iso["year"].values, _w=iso["week"].values)
                .groupby(["token_id", "_y", "_w"], sort=False, as_index=False)
                .head(1)
                .drop(columns=["_y", "_w"]))
        log("9 weekly: first day of ISO week per token", n, len(df))

    df = df.sort_values(["token_id", "date"], kind="stable").reset_index(drop=True)

    nulls = df[FULL + list(TARGETS)].isna().sum()
    if nulls.any():
        raise ValueError(f"nulls remain after cleaning:\n{nulls[nulls > 0]}")

    lead = ["token_id", "date"] + list(TARGETS)
    df = df[lead + [c for c in df.columns if c not in lead]]
    if verbose:
        print(f"[load_data] freq={freq}: {len(df):,d} rows, "
              f"{df['token_id'].nunique()} tokens, "
              f"{df['date'].min().date()} .. {df['date'].max().date()}")
    return df


# --------------------------------------------------------------------------
# Walk-forward folds
# --------------------------------------------------------------------------
def make_folds(df: pd.DataFrame, n_folds: int = 5, test_days: int = 90,
               embargo_days: int = 30, val_days: int = 90, verbose: bool = True):
    """Walk-forward folds by date, all tokens pooled.

    Returns a list of (train_idx, val_idx, test_idx) positional index arrays.
    val_idx is the last `val_days` of train and is a subset of train_idx.
    Cutoff dates are stored on the returned list as `.cutoffs`.
    """
    dates = df["date"].to_numpy()
    max_date = df["date"].max()
    day = pd.Timedelta(days=1)
    folds = FoldList()
    for i in range(n_folds):
        c = max_date - (n_folds - i) * test_days * day
        train_end = c - embargo_days * day
        val_start = train_end - val_days * day  # val = (val_start, train_end]
        train_mask = dates <= train_end.to_datetime64()
        val_mask = train_mask & (dates > val_start.to_datetime64())
        test_mask = (dates > c.to_datetime64()) & \
                    (dates <= (c + test_days * day).to_datetime64())
        tr, va, te = (np.flatnonzero(m) for m in (train_mask, val_mask, test_mask))
        if min(len(tr) - len(va), len(va), len(te)) <= 0:
            raise ValueError(f"fold {i + 1} has an empty slice; "
                             "reduce n_folds/test_days/val_days")

        # no test row within embargo_days after any train row
        gap = (dates[te].min() - dates[tr].max()) / np.timedelta64(1, "D")
        assert gap > embargo_days, f"fold {i + 1}: embargo violated (gap {gap}d)"
        assert not np.intersect1d(tr, te).size
        assert np.isin(va, tr).all()

        folds.append((tr, va, te))
        folds.cutoffs.append(c)
        if verbose:
            d = lambda ix: f"{pd.Timestamp(dates[ix].min()).date()}..{pd.Timestamp(dates[ix].max()).date()}"
            print(f"[make_folds] fold {i + 1} cutoff {c.date()} | "
                  f"train {len(tr):>6,d} {d(tr)} | "
                  f"val {len(va):>6,d} {d(va)} | "
                  f"test {len(te):>6,d} {d(te)}")
    return folds


class FoldList(list):
    """A list of (train_idx, val_idx, test_idx) that also carries cutoff dates."""

    def __init__(self, *args):
        super().__init__(*args)
        self.cutoffs: list[pd.Timestamp] = []


def fit_slice(df: pd.DataFrame, train_idx, val_idx, purge_days: int = 30):
    """Rows of train used to fit during tuning: train minus val, minus the
    `purge_days` right before val (their forward targets overlap the val
    period, which would flatter low-bias models such as small-k KNN)."""
    dates = df["date"].to_numpy()
    val_start = dates[val_idx].min()
    keep = dates[train_idx] <= val_start - np.timedelta64(purge_days + 1, "D")
    return np.asarray(train_idx)[keep]


# --------------------------------------------------------------------------
# Metrics and baselines
# --------------------------------------------------------------------------
def evaluate(y_true, y_pred, y_persist) -> dict:
    """Metrics in log-vol space. r2_oos is relative to the persistence
    forecast (y_persist = log rv_past_14d): 0 = ties persistence."""
    y_true, y_pred, y_persist = (np.asarray(a, dtype=float)
                                 for a in (y_true, y_pred, y_persist))
    err = y_true - y_pred
    sse = float(np.sum(err ** 2))
    sse_p = float(np.sum((y_true - y_persist) ** 2))
    rho = spearmanr(y_true, y_pred).statistic if np.ptp(y_pred) > 0 else np.nan
    return {
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mae": float(np.mean(np.abs(err))),
        "r2_oos": 1.0 - sse / sse_p,
        "spearman": float(rho),
        "rmse_level": float(np.sqrt(np.mean((np.exp(y_true) - np.exp(y_pred)) ** 2))),
        "n": int(len(y_true)),
    }


RESULT_COLUMNS = ["model", "feature_set", "freq", "target", "fold", "cutoff_date",
                  "n_train", "n_test", "best_params", "rmse", "mae", "r2_oos",
                  "spearman", "rmse_level", "subsample"]
RUN_KEY = ["model", "feature_set", "freq", "target", "subsample"]


def _upsert_results(path: Path, new: pd.DataFrame) -> None:
    """Append to a results CSV, replacing rows from an earlier identical run."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        old = pd.read_csv(path)
        if "subsample" not in old.columns:
            old["subsample"] = "all"
        keys = set(map(tuple, new[RUN_KEY].astype(str).drop_duplicates().to_numpy()))
        keep = [tuple(r) not in keys for r in old[RUN_KEY].astype(str).to_numpy()]
        new = pd.concat([old[keep], new], ignore_index=True)
    new[RESULT_COLUMNS].to_csv(path, index=False)


def baselines(df: pd.DataFrame, folds, freq: str = "daily", target: str = "y",
              subsample: str = "all", save: bool = True) -> pd.DataFrame:
    """Persistence (log rv_past_14d) and pooled train-mean forecasts per fold."""
    y = df[target].to_numpy()
    persist = np.log(df[PERSIST_COL].to_numpy())
    rows = []
    for k, (tr, _, te) in enumerate(folds, start=1):
        preds = {"persistence": persist[te],
                 "train_mean": np.full(len(te), y[tr].mean())}
        for name, p in preds.items():
            m = evaluate(y[te], p, persist[te])
            rows.append({"model": name, "feature_set": "NONE", "freq": freq,
                         "target": TARGETS[target], "fold": k,
                         "cutoff_date": folds.cutoffs[k - 1].date().isoformat(),
                         "n_train": len(tr), "n_test": m.pop("n"),
                         "best_params": "{}", **m, "subsample": subsample})
    out = pd.DataFrame(rows)[RESULT_COLUMNS]
    if save:
        _upsert_results(RESULTS_DIR / "baselines.csv", out)
    return out


# --------------------------------------------------------------------------
# Generic model loop
# --------------------------------------------------------------------------
def expand_grid(param_grid: dict, max_configs: int | None = None) -> list[dict]:
    """All combinations of param_grid; optionally a fixed random subset."""
    keys = list(param_grid)
    configs = [dict(zip(keys, v)) for v in itertools.product(*param_grid.values())]
    if max_configs is not None and max_configs < len(configs):
        rng = np.random.default_rng(RANDOM_STATE)
        pick = sorted(rng.choice(len(configs), size=max_configs, replace=False))
        configs = [configs[i] for i in pick]
    return configs


def _jsonable(params: dict) -> str:
    return json.dumps({k: (v.item() if isinstance(v, np.generic) else v)
                       for k, v in params.items()}, sort_keys=True)


def run_tag(feature_set_name, freq, target="y", subsample="all") -> str:
    tag = f"{feature_set_name}_{freq}"
    if target != "y":
        tag += f"_{TARGETS[target]}"
    if subsample != "all":
        tag += f"_{subsample}"
    return tag


def run_model(name, estimator_factory, df, folds, feature_set_name, param_grid,
              freq, *, target: str = "y", subsample: str = "all",
              max_configs: int | None = None, purge_days: int = 30,
              save: bool = True, verbose: bool = True) -> pd.DataFrame:
    """Tune on the validation slice, refit on full train, score on test.

    estimator_factory(**params) must return an unfitted sklearn Pipeline (or
    estimator); any scaler inside it is therefore fit on training rows only.
    param_grid is {param_name: [values]}; max_configs evaluates a fixed random
    subset of the grid (seeded) when the full grid is too slow.
    `df` must be the frame `folds` was built from. `subsample` is only a label
    for the results CSV -- filter df yourself before calling make_folds.

    Writes results/<name>.csv, results/preds_<name>_<tag>.parquet and
    results/val_<name>_<tag>.csv (validation RMSE of every config tried).
    """
    features = FEATURE_SETS[feature_set_name]
    if target not in TARGETS:
        raise KeyError(f"target must be one of {list(TARGETS)}")
    X = df[features].to_numpy(dtype=float)
    y = df[target].to_numpy()
    persist = np.log(df[PERSIST_COL].to_numpy())
    configs = expand_grid(param_grid, max_configs)
    tag = run_tag(feature_set_name, freq, target, subsample)

    rows, preds, val_rows = [], [], []
    t_start = time.perf_counter()
    for k, (tr, va, te) in enumerate(folds, start=1):
        t0 = time.perf_counter()
        fit = fit_slice(df, tr, va, purge_days)
        best = (np.inf, None)
        for params in configs:
            est = estimator_factory(**params).fit(X[fit], y[fit])
            rmse = float(np.sqrt(np.mean((y[va] - est.predict(X[va])) ** 2)))
            val_rows.append({"model": name, "feature_set": feature_set_name,
                             "freq": freq, "target": TARGETS[target],
                             "subsample": subsample, "fold": k, **params,
                             "val_rmse": rmse})
            if rmse < best[0]:
                best = (rmse, params)
        est = estimator_factory(**best[1]).fit(X[tr], y[tr])
        p = est.predict(X[te])
        m = evaluate(y[te], p, persist[te])
        rows.append({"model": name, "feature_set": feature_set_name, "freq": freq,
                     "target": TARGETS[target], "fold": k,
                     "cutoff_date": folds.cutoffs[k - 1].date().isoformat(),
                     "n_train": len(tr), "n_test": m.pop("n"),
                     "best_params": _jsonable(best[1]), **m,
                     "subsample": subsample})
        preds.append(pd.DataFrame({"token_id": df["token_id"].to_numpy()[te],
                                   "date": df["date"].to_numpy()[te],
                                   "y_true": y[te], "y_pred": p, "fold": k}))
        if verbose:
            print(f"[{name}|{tag}] fold {k}: val_rmse {best[0]:.4f} "
                  f"test rmse {m['rmse']:.4f} r2_oos {m['r2_oos']:+.4f} "
                  f"best {_jsonable(best[1])} ({time.perf_counter() - t0:.1f}s)")

    out = pd.DataFrame(rows)[RESULT_COLUMNS]
    if verbose:
        print(f"[{name}|{tag}] mean r2_oos {out['r2_oos'].mean():+.4f} "
              f"mean rmse {out['rmse'].mean():.4f} "
              f"({len(configs)} configs, {time.perf_counter() - t_start:.1f}s total)")
    if save:
        _upsert_results(RESULTS_DIR / f"{name}.csv", out)
        pd.concat(preds, ignore_index=True).to_parquet(
            RESULTS_DIR / f"preds_{name}_{tag}.parquet", index=False)
        pd.DataFrame(val_rows).to_csv(
            RESULTS_DIR / f"val_{name}_{tag}.csv", index=False)
    return out


# --------------------------------------------------------------------------
# Fixed log transform for the linear models (Ridge / Elastic Net / logistic)
# --------------------------------------------------------------------------
# Volatility levels and token age are right-skewed and enter a linear model
# better in logs. Nothing is learned from data, so it is safe in any fold.
LOG_COLS = ["rv_past_7d", "rv_past_14d", "rv_past_30d",
            "btc_rv_past_14d", "eth_rv_past_14d"]
LOG1P_COLS = ["days_since_tge"]


def _log_columns(X, ix_log=(), ix_log1p=()):
    X = np.array(X, dtype=float, copy=True)
    X[:, list(ix_log)] = np.log(X[:, list(ix_log)])
    X[:, list(ix_log1p)] = np.log1p(X[:, list(ix_log1p)])
    return X


def log_transformer(features: list[str]):
    """FunctionTransformer that logs LOG_COLS and log1ps LOG1P_COLS, located
    by position in `features` (the column order of the feature matrix)."""
    from sklearn.preprocessing import FunctionTransformer
    return FunctionTransformer(_log_columns, kw_args={
        "ix_log": [i for i, f in enumerate(features) if f in LOG_COLS],
        "ix_log1p": [i for i, f in enumerate(features) if f in LOG1P_COLS]})


# --------------------------------------------------------------------------
# Plot style shared by all scripts
# --------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"


def set_style() -> None:
    from cycler import cycler
    plt.rcParams.update({
        "figure.dpi": 100, "savefig.dpi": 150, "savefig.bbox": "tight",
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.prop_cycle": cycler(color=PALETTE),
        "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
        "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.labelsize": 10,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK_2, "ytick.color": INK_2,
        "xtick.labelsize": 9, "ytick.labelsize": 9,
        "text.color": INK, "legend.frameon": False, "legend.fontsize": 9,
        "lines.linewidth": 2, "lines.markersize": 6, "font.size": 10,
    })


def savefig(fig, name: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=150)
    display(fig)          # show inline in the notebook ...
    plt.close(fig)        # ... and save a PNG copy under figures/
    print(f"[figure] {path.relative_to(ROOT)}")
    return path




# ---- resume support -------------------------------------------------------------
# Every model run is ~5-45 min (Random Forest) and results/<model>.csv keeps one row
# per fold per run, so a finished run is reused instead of recomputed. Set
# FRESH = True (or delete the CSV) to recompute everything from scratch.
FRESH = os.environ.get("NB_FRESH") == "1"


def finished_run(model, folds, fs, freq, target="y", subsample="all"):
    """Rows already in results/<model>.csv for this run, or None if incomplete."""
    path = RESULTS_DIR / f"{model}.csv"
    if SMOKE or FRESH or not path.exists():
        return None
    old = pd.read_csv(path)
    key = {"model": model, "feature_set": fs, "freq": freq,
           "target": TARGETS[target], "subsample": subsample}
    mask = np.logical_and.reduce([old[k].astype(str) == v for k, v in key.items()])
    rows = old[mask]
    if len(rows) == len(folds) and sorted(rows["fold"]) == list(range(1, len(folds) + 1)):
        print(f"[{model}|{run_tag(fs, freq, target, subsample)}] found in results/{model}.csv, "
              f"reusing (mean rmse {rows['rmse'].mean():.4f}, r2_oos {rows['r2_oos'].mean():+.4f}); "
              "set NB_FRESH=1 to recompute")
        return rows.reset_index(drop=True)
    return None


## 1. Load and clean the panel

`load_data` parses dates, sorts by (token, date), drops rows without a target or 30 days of price history, takes logs of the targets and of the long-tailed emission ratios, fills the two structurally-null cliff columns, encodes the categorical columns with fixed category lists (nothing is learned from the data), and drops identifiers, raw token counts and every raw target so none can leak into a feature matrix. The weekly panel keeps the first available day of each ISO week per token.


In [ ]:
daily = load_data(freq="daily")
weekly = load_data(freq="weekly")
display(daily.head())
display(daily[["y"] + CONTROLS + EMIS5].describe().T.round(4))

## 2. Walk-forward folds

Five folds by calendar date with all tokens pooled. The printout shows, per fold, the cutoff and the date ranges of train, validation (last 90 days of train) and test (90 days after the cutoff, separated from train by a 30-day embargo).


In [ ]:
folds_daily = make_folds(daily)
folds_weekly = make_folds(weekly)
data = {"daily": (daily, folds_daily), "weekly": (weekly, folds_weekly)}
set_style()

## 3. Exploratory data analysis

Distribution of the target and the main emission feature, forward volatility by decile of unlock pressure, Spearman correlations between target, controls and emission features, and per-token coverage against the fold cutoffs.


In [ ]:
rv = np.exp(daily["y"])                                # rv_fwd_14d, level
emis = np.expm1(daily["emis_next_30d_pct_float"])      # raw fraction of float

# ---- 1. histograms, raw and log ------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
panels = [
    (rv, "rv_fwd_14d (raw, annualized)", PALETTE[0]),
    (daily["y"], "log(rv_fwd_14d)  = target y", PALETTE[0]),
    (emis, "emis_next_30d_pct_float (raw)", PALETTE[1]),
    (daily["emis_next_30d_pct_float"], "log1p(emis_next_30d_pct_float)", PALETTE[1]),
]
for ax, (s, label, color) in zip(axes.ravel(), panels):
    ax.hist(s, bins=80, color=color, edgecolor=SURFACE, linewidth=0.3)
    ax.set_title(label)
    ax.set_ylabel("rows")
    ax.text(0.98, 0.95, f"median {s.median():.3f}\nmean {s.mean():.3f}\n"
            f"max {s.max():.2f}\nzeros {(s == 0).mean():.1%}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8, color=INK_2)
axes[1, 0].set_yscale("log")
axes[1, 1].set_yscale("log")
axes[1, 0].set_ylabel("rows (log scale)")
axes[1, 1].set_ylabel("rows (log scale)")
fig.suptitle("Target and main emission feature: raw vs log (daily panel, "
             f"{len(daily):,d} rows)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
savefig(fig, "eda_histograms.png")


In [ ]:
# ---- 2. decile plot -------------------------------------------------------
dec = pd.qcut(emis.rank(method="first"), 10, labels=False) + 1
g = pd.DataFrame({"decile": dec, "rv": rv, "emis": emis}).groupby("decile")
stats = g["rv"].agg(mean="mean", median="median",
                    q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
stats["emis_median"] = g["emis"].median()
fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(stats.index, stats["q25"], stats["q75"], color=PALETTE[0],
                alpha=0.18, linewidth=0, label="IQR (25th-75th pct)")
ax.plot(stats.index, stats["median"], color=PALETTE[0], linestyle="--",
        linewidth=1.5, label="median")
ax.plot(stats.index, stats["mean"], color=PALETTE[0], marker="o", label="mean")
ax.set_xticks(stats.index)
ax.set_xticklabels([f"D{d}\n{m:.1%}" for d, m in stats["emis_median"].items()])
ax.set_xlabel("decile of emis_next_30d_pct_float (median share of float unlocking in 30d)")
ax.set_ylabel("rv_fwd_14d (annualized)")
ax.set_ylim(0, None)
ax.set_title("Forward 14d volatility by unlock-pressure decile (daily panel)")
ax.legend(loc="upper left")
savefig(fig, "eda_decile_emission_vs_rv.png")
print(stats.round(4).to_string())


In [ ]:
# ---- 3. correlation heatmap ----------------------------------------------
top_emis = ["emis_next_7d_pct_float", "emis_next_30d_pct_float",
            "emis_next_90d_pct_float", "emis_next_30d_pct_total",
            "days_to_next_cliff", "next_cliff_pct_float", "has_unlock_events"]
cols = ["y"] + CONTROLS + top_emis
corr = daily[cols].corr(method="spearman")
cmap = LinearSegmentedColormap.from_list("div", ["#d03b3b", "#f0efec", "#2a78d6"])
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap=cmap, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7}, linewidths=1, linecolor=SURFACE,
            cbar_kws={"label": "Spearman rho", "shrink": 0.7}, ax=ax)
ax.grid(False)
ax.set_title("Spearman correlation: target y, controls and emission features "
             "(daily panel)")
savefig(fig, "eda_correlation_heatmap.png")


In [ ]:
# ---- 4. per-token coverage ------------------------------------------------
cov = daily.groupby("token_id")["date"].agg(["count", "min", "max"]).sort_values(
    ["min", "max"], ascending=[True, False])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(cov["count"], bins=40, color=PALETTE[0], edgecolor=SURFACE)
ax.axvline(cov["count"].median(), color=INK_2, linestyle="--", linewidth=1)
ax.text(cov["count"].median(), ax.get_ylim()[1] * 0.95,
        f" median {cov['count'].median():.0f} rows", color=INK_2, fontsize=9,
        ha="right")
ax.set_xlabel("daily rows per token (after cleaning)")
ax.set_ylabel("tokens")
ax.set_title(f"Coverage: rows per token ({len(cov)} tokens)")
savefig(fig, "eda_rows_per_token.png")

fig, ax = plt.subplots(figsize=(9, 9))
ypos = np.arange(len(cov))
ax.hlines(ypos, cov["min"], cov["max"], color=PALETTE[0], linewidth=1.2)
folds = folds_daily
for c in folds.cutoffs:
    ax.axvline(c, color=INK_2, linestyle=":", linewidth=1)
ax.text(folds.cutoffs[0], len(cov) + 1, "fold cutoffs ", color=INK_2,
        fontsize=8, ha="right")
ax.set_yticks([])
ax.set_ylabel(f"{len(cov)} tokens, sorted by first date")
ax.set_ylim(-2, len(cov) + 4)
ax.grid(axis="y", visible=False)
ax.set_title("Coverage: date range of each token in the cleaned daily panel")
savefig(fig, "eda_token_date_ranges.png")


### 3.5 Summary table

Row counts, date range, the pooled and per-token Spearman correlation between the 30-day unlock-pressure feature and the target, and null rates after cleaning (all zero).


In [ ]:
# ---- 5. summary table -----------------------------------------------------
raw = pd.read_csv(DATA_PATH, usecols=["days_to_next_cliff", "unlockPrecision"])
pooled = spearmanr(daily["emis_next_30d_pct_float"], daily["y"]).statistic
per_tok = daily.groupby("token_id").apply(
    lambda d: spearmanr(d["emis_next_30d_pct_float"], d["y"]).statistic
    if d["emis_next_30d_pct_float"].nunique() > 1 else np.nan,
    include_groups=False)
rows = [
    ("rows_daily", len(daily)), ("rows_weekly", len(weekly)),
    ("tokens", daily["token_id"].nunique()),
    ("date_min", daily["date"].min().date()), ("date_max", daily["date"].max().date()),
    ("rows_per_token_median", cov["count"].median()),
    ("rows_per_token_min", cov["count"].min()),
    ("spearman_emis30_y_pooled", round(pooled, 4)),
    ("spearman_emis30_y_per_token_median", round(per_tok.median(), 4)),
    ("spearman_emis30_y_per_token_n", int(per_tok.notna().sum())),
    ("spearman_emis30_y_per_token_share_positive",
     round((per_tok.dropna() > 0).mean(), 4)),
    ("raw_null_rate_days_to_next_cliff (filled 365)",
     round(raw["days_to_next_cliff"].isna().mean(), 4)),
    ("raw_null_rate_unlockPrecision (-> unknown)",
     round(raw["unlockPrecision"].isna().mean(), 4)),
]
rows += [(f"null_rate_after_cleaning:{c}", round(daily[c].isna().mean(), 4))
         for c in FULL]
summary = pd.DataFrame(rows, columns=["metric", "value"])
if not SMOKE:
    RESULTS_DIR.mkdir(exist_ok=True)
    summary.to_csv(RESULTS_DIR / "eda_summary.csv", index=False)
print(summary.head(13).to_string(index=False))
print(f"max null rate after cleaning: {daily[FULL].isna().mean().max():.4f}")


## 4. Baselines

Two forecasts that use no model: **persistence** (`log rv_past_14d`, the benchmark inside `r2_oos`) and the **pooled train mean**. Note that the train mean already beats persistence, so every model is compared against both.


In [ ]:
baseline_res = pd.concat([baselines(df, folds, freq=freq, save=not SMOKE)
                          for freq, (df, folds) in data.items()], ignore_index=True)
display(baseline_res.groupby(["freq", "model"])[["rmse", "mae", "r2_oos", "spearman"]].mean().round(4))

## 5. K-nearest-neighbours regression

A `QuantileTransformer` (fit on training rows only) maps every feature to a normal scale so that distances are comparable, followed by `KNeighborsRegressor`. The grid is `k in {5, 10, 25, 50, 100, 250}` x `{uniform, distance}` weights. Three feature sets on both panels.


In [ ]:
KNN_NAME = "knn"
KNN_FEATURE_SETS = ["CONTROLS", "CONTROLS+EMIS5", "FULL"]
KNN_GRID = {"n_neighbors": [5, 10, 25, 50, 100, 250],
              "weights": ["uniform", "distance"]}
N_SHOWN = 15  # neighbours drawn in the illustration (the table has all k)


def knn_factory(**params) -> Pipeline:
    return Pipeline([
        ("qt", QuantileTransformer(output_distribution="normal",
                                   n_quantiles=1000,
                                   random_state=RANDOM_STATE)),
        ("knn", KNeighborsRegressor(algorithm="brute", n_jobs=-1, **params)),
    ])


In [ ]:
knn_results = []
for freq, (df, folds) in data.items():
    for fs in KNN_FEATURE_SETS:
        grid = {"n_neighbors": [25, 100], "weights": ["uniform"]} if SMOKE else KNN_GRID
        done = finished_run(KNN_NAME, folds, fs, freq)
        knn_results.append(done if done is not None else
                           run_model(KNN_NAME, knn_factory, df, folds, fs, grid, freq, save=not SMOKE))
knn_res = pd.concat(knn_results, ignore_index=True)

knn_summary = (knn_res.groupby(["freq", "feature_set"], sort=False)
                      [["rmse", "mae", "r2_oos", "spearman"]].mean())
print("KNN mean over folds:")
display(knn_summary.round(4))
for freq in data:
    a = knn_summary.loc[(freq, "CONTROLS"), "rmse"]
    c = knn_summary.loc[(freq, "FULL"), "rmse"]
    verdict = "DEGRADES" if c > a else "improves"
    print(f"[{freq}] RMSE CONTROLS {a:.4f} -> FULL {c:.4f}: adding emission+static "
          f"features {verdict} KNN (a result, not a bug)")


### 5.1 Validation curve and a neighbour illustration

The validation curve shows RMSE against k for each feature set (mean over folds, better of the two weightings). The illustration picks the most unlock-heavy test row of the last fold and shows its nearest training rows: who the model thinks the comparable token-days are, and what happened to them.


In [ ]:
def plot_val_curve():
    """Validation RMSE vs k (daily), one line per feature set. Each point is
    the mean over folds, using the better of the two weightings at that k."""
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, fs in enumerate(KNN_FEATURE_SETS):
        val = pd.read_csv(RESULTS_DIR / f"val_{KNN_NAME}_{run_tag(fs, 'daily')}.csv")
        curve = (val.groupby(["n_neighbors", "weights"])["val_rmse"].mean()
                    .unstack("weights").min(axis=1))
        ax.plot(curve.index, curve.values, marker="o", color=PALETTE[i],
                label=f"{fs} ({len(FEATURE_SETS[fs])} features), "
                      f"best {curve.min():.4f} at k={curve.idxmin()}")
    ax.set_xscale("log")
    ax.set_xticks(KNN_GRID["n_neighbors"])
    ax.set_xticklabels(KNN_GRID["n_neighbors"])
    ax.minorticks_off()
    ax.set_xlabel("k (n_neighbors, log scale)")
    ax.set_ylabel("validation RMSE, log-vol (mean of 5 folds)")
    ax.set_title("KNN: validation RMSE vs k by feature set (daily panel)")
    ax.legend(title="feature set")
    savefig(fig, "knn_val_rmse_vs_k.png")


def neighbour_illustration(df, folds, res, fs="CONTROLS+EMIS5"):
    """Who are the neighbours of the most unlock-heavy test row?"""
    features = FEATURE_SETS[fs]
    tr, _, te = folds[-1]
    row = res[(res.feature_set == fs) & (res.freq == "daily")
              & (res.fold == len(folds))].iloc[0]
    params = json.loads(row["best_params"])
    pipe = knn_factory(**params).fit(df.iloc[tr][features].to_numpy(float),
                                     df.iloc[tr]["y"].to_numpy())

    q_pos = te[np.argmax(df["emis_next_30d_pct_float"].to_numpy()[te])]
    q = df.iloc[[q_pos]]
    xq = pipe.named_steps["qt"].transform(q[features].to_numpy(float))
    dist, idx = pipe.named_steps["knn"].kneighbors(xq)
    nb = df.iloc[tr[idx[0]]]

    def table(d, role, distance):
        return pd.DataFrame({
            "role": role, "token_id": d["token_id"].values,
            "date": d["date"].dt.date.values, "distance": distance,
            "emis_next_30d_pct_float": np.expm1(d["emis_next_30d_pct_float"].values),
            "rv_past_14d": d["rv_past_14d"].values,
            "rv_fwd_14d": np.exp(d["y"].values)})

    out = pd.concat([table(q, "query", 0.0), table(nb, "neighbour", dist[0])],
                    ignore_index=True)
    pred = float(np.exp(pipe.predict(q[features].to_numpy(float))[0]))
    out["knn_pred_rv_fwd_14d"] = pred
    out.to_csv(RESULTS_DIR / "knn_neighbours.csv", index=False)
    print(f"\nNeighbour illustration ({fs}, fold {len(folds)}, {params}); "
          f"KNN prediction {pred:.3f} vs realized {out.loc[0, 'rv_fwd_14d']:.3f}")
    print(out.head(N_SHOWN + 1).round(4).to_string(index=False))

    shown = out.head(N_SHOWN + 1)
    labels = ["QUERY\n" + shown.loc[0, "token_id"]] + [
        f"{t}\n{d:%y-%m-%d}" for t, d in zip(shown["token_id"][1:], shown["date"][1:])]
    colors = [PALETTE[1]] + [PALETTE[0]] * (len(shown) - 1)
    panels = [("emis_next_30d_pct_float", "share of float unlocking in next 30d"),
              ("rv_past_14d", "rv_past_14d (known at D)"),
              ("rv_fwd_14d", "realized rv_fwd_14d (target, level)")]
    fig, axes = plt.subplots(3, 1, figsize=(12, 9.5), sharex=True)
    for ax, (col, title) in zip(axes, panels):
        ax.bar(range(len(shown)), shown[col], color=colors, width=0.7)
        ax.set_title(title, fontsize=10)
        ax.grid(axis="x", visible=False)
        ax.text(0, shown[col].iloc[0], f"{shown[col].iloc[0]:.2f}", ha="center",
                va="bottom", fontsize=8, color=INK_2)
    axes[2].axhline(pred, color=INK_2, linestyle="--", linewidth=1.2)
    axes[2].set_title(f"realized rv_fwd_14d (target, level); dashed = KNN "
                      f"prediction {pred:.2f} from all k={params['n_neighbors']} "
                      f"neighbours ({params['weights']} weights)", fontsize=10)
    axes[2].set_xticks(range(len(shown)))
    axes[2].set_xticklabels(labels, fontsize=7)
    qd = shown.loc[0, "date"]
    fig.suptitle(
        f"KNN neighbours of the most unlock-heavy test row: "
        f"{shown.loc[0, 'token_id']} on {qd} (orange) vs its {len(shown) - 1} "
        f"nearest training rows (blue, nearest first)\nfeature set {fs}, "
        f"last fold", x=0.01, ha="left", fontweight="bold", fontsize=11)
    fig.tight_layout()
    savefig(fig, "knn_neighbour_illustration.png")


In [ ]:
if not SMOKE:
    plot_val_curve()          # reads results/val_knn_*.csv written by run_model
neighbour_illustration(*data["daily"], knn_res)


## 6. Random Forest

400 trees; grid over `max_depth`, `min_samples_leaf`, `max_features` (36 configs). Following the project rule of subsampling the grid rather than dropping the protocol, the daily runs evaluate a fixed, seeded random subset of the grid (12 configs for the main runs, 6 for robustness runs); the weekly runs use the full grid. The fold protocol is unchanged.

### 6.1 Main runs: CONTROLS vs FULL, weekly and daily


In [ ]:
RF_NAME = "rf"
N_TREES = 30 if SMOKE else 400
RF_GRID = {"max_depth": [None, 8, 12, 16],
              "min_samples_leaf": [20, 50, 100],
              "max_features": ["sqrt", 0.33, 0.5]}
# configs evaluated per fold (None = full grid of 36)
BUDGET = {"daily_main": 12, "daily_robust": 6, "weekly": None}
if SMOKE:
    BUDGET = {k: 2 for k in BUDGET}

GROUP_COLOR = {"controls": PALETTE[0], "emission": PALETTE[1],
               "static": PALETTE[2]}


def group_of(feature: str) -> str:
    return ("controls" if feature in CONTROLS else
            "emission" if feature in EMISSION else "static")


def rf_factory(**params) -> Pipeline:
    # trees need no scaling; the Pipeline wrapper keeps the harness contract
    return Pipeline([("rf", RandomForestRegressor(
        n_estimators=N_TREES, n_jobs=-1, random_state=RANDOM_STATE, **params))])


def run(df, folds, fs, freq, budget, **kw):
    done = finished_run(RF_NAME, folds, fs, freq, **kw)
    if done is not None:
        return done
    return run_model(RF_NAME, rf_factory, df, folds, fs, RF_GRID, freq,
                       max_configs=BUDGET[budget], save=not SMOKE, **kw)

def main_runs():
    out = []
    for freq in ("weekly", "daily"):
        df, folds = data[freq]
        for fs in ("CONTROLS", "FULL"):
            out.append(run(df, folds, fs, freq,
                           "weekly" if freq == "weekly" else "daily_main"))
    return pd.concat(out)


def robustness_runs(df, folds):
    out = []
    for target in ("y_14d_parkinson", "y_7d", "y_30d"):
        baselines(df, folds, freq="daily", target=target, save=not SMOKE)
        out.append(run(df, folds, "FULL", "daily", "daily_robust", target=target))

    sub = df[df["listedMethod_INTERNAL"] == 1].reset_index(drop=True)
    print(f"[subsample INTERNAL] {len(sub):,d} rows, "
          f"{sub['token_id'].nunique()} tokens")
    sub_folds = make_folds(sub)
    baselines(sub, sub_folds, freq="daily", subsample="INTERNAL", save=not SMOKE)
    # controls-only on the same subsample, so the FULL number has a reference
    for fs in ("CONTROLS", "FULL"):
        out.append(run(sub, sub_folds, fs, "daily", "daily_robust",
                       subsample="INTERNAL"))
    return pd.concat(out)


In [ ]:
rf_main_res = main_runs()


### 6.2 Robustness

The same comparison on three alternative targets (Parkinson 14-day, 7-day and 30-day forward volatility) and on the subsample of tokens that Tokenomist curates by hand (`listedMethod == INTERNAL`), with a controls-only run on the same subsample as reference.


In [ ]:
rf_robust_res = robustness_runs(*data["daily"])
rf_res = pd.concat([rf_main_res, rf_robust_res], ignore_index=True)
print("RF mean over folds:")
display(rf_res.groupby(["freq", "target", "subsample", "feature_set"], sort=False)
              [["rmse", "mae", "r2_oos", "spearman"]].mean().round(4))


### 6.3 Interpretation (last fold, FULL config)

Permutation importance on the test set (increase in RMSE when a feature is shuffled, 10 repeats) next to impurity importance; the same for a variant that removes the aggregate 30-day feature so the six recipient-split columns have to carry the signal; and partial dependence of the prediction on 30-day unlock pressure, alone and against the BTC volatility regime.


In [ ]:
def importances(est, X_te, y_te, features, variant):
    pi = permutation_importance(est, X_te, y_te, n_repeats=10,
                                random_state=RANDOM_STATE,
                                scoring="neg_root_mean_squared_error")
    return pd.DataFrame({
        "model_variant": variant, "feature": features,
        "group": [group_of(f) for f in features],
        "perm_importance_mean": pi.importances_mean,
        "perm_importance_std": pi.importances_std,
        "impurity_importance": est.named_steps["rf"].feature_importances_,
    }).sort_values("perm_importance_mean", ascending=False)


def interpret(df, folds, params):
    tr, _, te = folds[-1]
    y = df["y"].to_numpy()
    print(f"[interpret] last fold, FULL config {params}")

    fitted, imps = {}, []
    for variant in ("FULL", "FULL_NO_AGG30"):
        feats = FEATURE_SETS[variant]
        t0 = time.perf_counter()
        X = df[feats].to_numpy(float)
        est = rf_factory(**params).fit(X[tr], y[tr])
        rmse = float(np.sqrt(np.mean((y[te] - est.predict(X[te])) ** 2)))
        imps.append(importances(est, X[te], y[te], feats, variant))
        fitted[variant] = (est, X)
        print(f"[interpret] {variant}: test rmse {rmse:.4f} "
              f"({time.perf_counter() - t0:.1f}s incl. permutation importance)")
    imp = pd.concat(imps, ignore_index=True)
    imp.to_csv(RESULTS_DIR / "rf_permutation_importance.csv", index=False)
    full = imp[imp.model_variant == "FULL"]
    print(full.head(15).round(5).to_string(index=False))

    plot_importance(full)
    plot_recipient_split(imp)
    plot_pdp(*fitted["FULL"], te)


def plot_importance(full):
    top = full.head(15).iloc[::-1]
    colors = [GROUP_COLOR[g] for g in top["group"]]
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
    axes[0].barh(top["feature"], top["perm_importance_mean"],
                 xerr=top["perm_importance_std"], color=colors, height=0.7,
                 error_kw={"ecolor": INK_2, "elinewidth": 1})
    axes[0].axvline(0, color=INK_2, linewidth=0.8)
    axes[0].set_title("Permutation importance on the TEST set\n"
                      "(increase in RMSE when shuffled, 10 repeats)", fontsize=10)
    axes[0].set_xlabel("increase in test RMSE (log-vol)")
    axes[1].barh(top["feature"], top["impurity_importance"], color=colors,
                 height=0.7)
    axes[1].set_title("Impurity importance (training set)\n"
                      "same features, same order", fontsize=10)
    axes[1].set_xlabel("share of impurity reduction")
    for ax in axes:
        ax.grid(axis="y", visible=False)
    axes[1].legend(handles=[Patch(color=c, label=g) for g, c in GROUP_COLOR.items()],
                   title="feature group", loc="lower right")
    fig.suptitle("Random Forest (FULL, last fold): top 15 features by "
                 "permutation importance", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    savefig(fig, "rf_permutation_importance_top15.png")


def plot_recipient_split(imp):
    piv = (imp[imp.feature.isin(EMISSION_BUCKETS + ["emis_next_30d_pct_float"])]
           .pivot(index="feature", columns="model_variant",
                  values=["perm_importance_mean", "perm_importance_std"]))
    order = ["emis_next_30d_pct_float"] + EMISSION_BUCKETS
    piv = piv.reindex(order)
    labels = ["aggregate\n(all recipients)"] + BUCKETS
    x = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (variant, label) in enumerate(
            [("FULL", "FULL (aggregate included)"),
             ("FULL_NO_AGG30", "recipient split (aggregate removed)")]):
        ax.bar(x + (i - 0.5) * 0.38, piv[("perm_importance_mean", variant)],
               yerr=piv[("perm_importance_std", variant)], width=0.36,
               color=PALETTE[i], label=label,
               error_kw={"ecolor": INK_2, "elinewidth": 1})
    ax.axhline(0, color=INK_2, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.grid(axis="x", visible=False)
    ax.set_ylabel("permutation importance\n(increase in test RMSE, log-vol)")
    ax.set_xlabel("emis_next_30d_<recipient>_pct_float")
    ax.set_title("Permutation importance of 30-day emission by recipient "
                 "(RF, last fold)", fontsize=11)
    ax.legend()
    savefig(fig, "rf_recipient_split_importance.png")
    print(piv["perm_importance_mean"].round(5).to_string())


def plot_pdp(est, X, te):
    feats = FULL
    i_e, i_b = feats.index("emis_next_30d_pct_float"), feats.index("btc_rv_past_14d")
    rng = np.random.default_rng(RANDOM_STATE)
    Xs = X[np.sort(rng.choice(te, size=min(3000, len(te)), replace=False))]

    pd1 = partial_dependence(est, Xs, [i_e], percentiles=(0, 0.99),
                             grid_resolution=40, kind="average")
    gx = np.expm1(pd1["grid_values"][0])
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(gx * 100, pd1["average"][0], color=PALETTE[0])
    deciles = np.expm1(np.quantile(X[te, i_e], np.linspace(0.1, 0.9, 9))) * 100
    ax.plot(deciles, np.full(9, ax.get_ylim()[0]), "|", color=INK_2,
            markersize=12, clip_on=False)
    ax.set_xlabel("share of float unlocking in the next 30 days, % "
                  "(ticks at the bottom = test-set deciles)")
    ax.set_ylabel("partial dependence: predicted log(rv_fwd_14d)")
    span = pd1["average"][0].max() - pd1["average"][0].min()
    ax.set_title("RF partial dependence on 30-day unlock pressure (FULL, last "
                 f"fold)\ntotal range of the curve: {span:.3f} log-vol",
                 fontsize=11)
    savefig(fig, "rf_pdp_emis30.png")

    pd2 = partial_dependence(est, Xs, [(i_e, i_b)], percentiles=(0, 0.99),
                             grid_resolution=20, kind="average")
    ge, gb = pd2["grid_values"]
    cmap = LinearSegmentedColormap.from_list(
        "seq", ["#cde2fb", "#6da7ec", "#2a78d6", "#0d366b"])
    fig, ax = plt.subplots(figsize=(9, 6))
    cs = ax.contourf(np.expm1(ge) * 100, gb, pd2["average"][0].T, levels=12,
                     cmap=cmap)
    ax.contour(np.expm1(ge) * 100, gb, pd2["average"][0].T, levels=12,
               colors=SURFACE, linewidths=0.5, negative_linestyles="solid")
    fig.colorbar(cs, ax=ax, label="predicted log(rv_fwd_14d)")
    ax.grid(False)
    ax.set_xlabel("share of float unlocking in the next 30 days, %")
    ax.set_ylabel("btc_rv_past_14d (market vol regime)")
    ax.set_title("RF 2-D partial dependence: unlock pressure x BTC vol regime",
                 fontsize=11)
    savefig(fig, "rf_pdp_emis30_x_btc.png")


In [ ]:
df_d, folds_d = data["daily"]
best = rf_res[(rf_res.feature_set == "FULL") & (rf_res.freq == "daily")
              & (rf_res.target == TARGETS["y"]) & (rf_res.subsample == "all")
              & (rf_res.fold == len(folds_d))]
interpret(df_d, folds_d, json.loads(best.iloc[0]["best_params"]))


## 7. Ridge and Elastic Net regression

Linear benchmarks on the same rows, features, folds and tuning protocol as KNN and RF. Inside each `Pipeline` the volatility controls (`rv_past_*`, BTC/ETH vol) are logged and `days_since_tge` is log1p'd (a fixed transform, nothing learned), then every feature is standardized on the training slice only. Grids: Ridge `alpha in {0.1 .. 1e7}` (9 values); Elastic Net `alpha in {1e-4 .. 1}` (9; at 1 every coefficient is zero, i.e. the train mean) x `l1_ratio in {0.1, 0.5, 0.9}`. Both models are run on CONTROLS and FULL, on both panels, on the three robustness targets and on the INTERNAL subsample.

### 7.1 Runs

In [ ]:
RIDGE_GRID = {"alpha": [0.1, 1, 10, 100, 1e3, 1e4, 1e5, 1e6, 1e7]}
ENET_GRID = {"alpha": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0],
             "l1_ratio": [0.1, 0.5, 0.9]}
if SMOKE:
    RIDGE_GRID = {"alpha": [10, 1e4]}
    ENET_GRID = {"alpha": [1e-3], "l1_ratio": [0.5]}
LINEAR_MODELS = {"ridge": RIDGE_GRID, "enet": ENET_GRID}


def linear_factory(model: str, features: list[str]):
    """estimator_factory for run_model: log -> standardize -> Ridge / EN."""
    def factory(**params) -> Pipeline:
        reg = (Ridge(**params) if model == "ridge" else
               ElasticNet(max_iter=5000, precompute=True,  # Gram matrix: n >> p, ~100x faster
                          random_state=RANDOM_STATE, **params))
        return Pipeline([("log", log_transformer(features)),
                         ("scale", StandardScaler()),
                         (model, reg)])
    return factory


def run_linear(model, df, folds, fs, freq, **kw):
    done = finished_run(model, folds, fs, freq, **kw)
    if done is not None:
        return done
    return run_model(model, linear_factory(model, FEATURE_SETS[fs]), df, folds,
                     fs, LINEAR_MODELS[model], freq, save=not SMOKE, **kw)


linear_results = []
for freq, (df, folds) in data.items():
    for model in LINEAR_MODELS:
        for fs in ("CONTROLS", "FULL"):
            linear_results.append(run_linear(model, df, folds, fs, freq))

df_d, folds_d = data["daily"]
for target in ("y_14d_parkinson", "y_7d", "y_30d"):
    for model in LINEAR_MODELS:
        for fs in ("CONTROLS", "FULL"):
            linear_results.append(run_linear(model, df_d, folds_d, fs, "daily",
                                             target=target))
sub = df_d[df_d["listedMethod_INTERNAL"] == 1].reset_index(drop=True)
sub_folds = make_folds(sub, verbose=False)
for model in LINEAR_MODELS:
    for fs in ("CONTROLS", "FULL"):
        linear_results.append(run_linear(model, sub, sub_folds, fs, "daily",
                                         subsample="INTERNAL"))
linear_res = pd.concat(linear_results, ignore_index=True)

print("Ridge / Elastic Net mean over folds:")
display(linear_res.groupby(["model", "freq", "target", "subsample", "feature_set"],
                           sort=False)[["rmse", "mae", "r2_oos", "spearman"]]
        .mean().round(4))

### 7.2 Coefficients (FULL, daily)

Each fold's tuned model is refit on its training slice and its coefficients on the standardized features are collected. Bars are the mean over the five folds, whiskers the min and max. Coefficients are in log-vol per one standard deviation of the feature, so they are comparable across features; the colour shows the feature group.

In [ ]:
def linear_coefficients(df, folds, res):
    """Refit FULL (daily) with each fold's tuned params; standardized coefs."""
    feats = FULL
    X, y = df[feats].to_numpy(float), df["y"].to_numpy()
    rows = []
    for model in LINEAR_MODELS:
        r = res[(res.model == model) & (res.feature_set == "FULL")
                & (res.freq == "daily") & (res.target == TARGETS["y"])
                & (res.subsample == "all")].set_index("fold")
        for k, (tr, _, _) in enumerate(folds, start=1):
            params = json.loads(r.loc[k, "best_params"])
            est = linear_factory(model, feats)(**params).fit(X[tr], y[tr])
            for f, c in zip(feats, est.named_steps[model].coef_):
                rows.append({"model": model, "fold": k, "feature": f,
                             "group": group_of(f), "coef_std": c,
                             "best_params": json.dumps(params, sort_keys=True)})
    coefs = pd.DataFrame(rows)
    if not SMOKE:
        coefs.to_csv(RESULTS_DIR / "linear_coefs.csv", index=False)
    return coefs


def plot_linear_coefs(coefs):
    agg = (coefs.groupby(["model", "feature"])["coef_std"]
                .agg(["mean", "min", "max"]).reset_index())
    order = (agg[agg.model == "ridge"].assign(a=lambda d: d["mean"].abs())
             .sort_values("a")["feature"].tolist())
    fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)
    for ax, model in zip(axes, LINEAR_MODELS):
        a = agg[agg.model == model].set_index("feature").reindex(order)
        colors = [GROUP_COLOR[group_of(f)] for f in order]
        ax.barh(order, a["mean"], color=colors, height=0.7,
                xerr=[a["mean"] - a["min"], a["max"] - a["mean"]],
                error_kw={"ecolor": INK_2, "elinewidth": 1})
        ax.axvline(0, color=INK_2, linewidth=0.8)
        ax.grid(axis="y", visible=False)
        n_zero = int((coefs[coefs.model == model].groupby("feature")["coef_std"]
                      .apply(lambda s: (s == 0).all())).sum())
        sub = f", {n_zero} features zeroed in every fold" if model == "enet" else ""
        ax.set_title(f"{model}: mean over 5 folds, bar = min..max{sub}",
                     fontsize=10)
        ax.set_xlabel("coefficient on standardized feature (log-vol per 1 SD)")
    axes[1].legend(handles=[Patch(color=c, label=g) for g, c in GROUP_COLOR.items()],
                   title="feature group", loc="lower right")
    fig.suptitle("Ridge and Elastic Net coefficients, FULL feature set "
                 "(daily, target log rv_fwd_14d)", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    savefig(fig, "linear_coefs_full.png")


linear_coefs = linear_coefficients(df_d, folds_d, linear_res)
plot_linear_coefs(linear_coefs)
display(linear_coefs[linear_coefs.model == "ridge"].groupby("feature")["coef_std"]
        .mean().sort_values(key=np.abs, ascending=False).head(12).round(4)
        .to_frame("ridge mean coef_std"))

## 8. Logistic regression: high-volatility events

A classification view of the same question. Two events, both defined from the forward target (labels, never inputs):
- **`top20_xs`**: `rv_fwd_14d` in the top 20% of all tokens on date D (cross-section: which tokens will be the most volatile?). Base rate 20% by construction.
- **`spike_1.5x`**: `rv_fwd_14d > 1.5 x rv_past_30d` (time series: will this token's volatility jump against its own recent level?). Base rate about 13%.

Labels are built on the daily panel, so the cross-sectional quantile uses every token on that date; the weekly panel picks them up by (token, date). Same rows, features, folds and protocol as the regressions: `C in {1e-5 .. 10}` is tuned on the validation slice by log loss, refit on all of train, scored once on test. Pipeline: the same fixed log transform, `StandardScaler`, L2 `LogisticRegression`. No resampling: the metrics are threshold-free (ROC-AUC, PR-AUC) or proper scores (Brier, log loss), and `brier_skill` is relative to the training base rate. For `top20_xs` the naive reference ranks tokens by `rv_past_14d`.

These results have their own file (`results/logistic.csv`, no RMSE), so they are reported here and not in the regression comparison below.

In [ ]:
LOGIT_NAME = "logistic"
LOGIT_C_GRID = [1e-3, 1] if SMOKE else [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10]
LABELS = ["top20_xs", "spike_1.5x"]
EPS = 1e-6
LOGIT_COLUMNS = ["model", "feature_set", "freq", "label", "fold", "cutoff_date",
                 "n_train", "n_test", "best_params", "auc", "pr_auc", "brier",
                 "log_loss", "brier_skill", "base_rate", "subsample"]
LOGIT_KEY = ["model", "feature_set", "freq", "label", "subsample"]


def add_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Event labels from the forward target (daily panel)."""
    q80 = df.groupby("date")["y"].transform(lambda s: s.quantile(0.8))
    return df.assign(**{
        "top20_xs": (df["y"] > q80).astype(int),
        "spike_1.5x": (df["y"] > np.log(1.5 * df["rv_past_30d"])).astype(int)})


def logit_factory(features, C_):
    return Pipeline([("log", log_transformer(features)),
                     ("scale", StandardScaler()),
                     ("logit", LogisticRegression(C=C_, max_iter=2000,
                                                  random_state=RANDOM_STATE))])


def logit_scores(y, p, base_rate=None, prob=True) -> dict:
    out = {"auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
           "brier": np.nan, "log_loss": np.nan, "brier_skill": np.nan,
           "base_rate": float(np.mean(y))}
    if prob:
        p = np.clip(p, EPS, 1 - EPS)
        out["brier"] = brier_score_loss(y, p)
        out["log_loss"] = log_loss(y, p, labels=[0, 1])
        if base_rate is not None:
            ref = brier_score_loss(y, np.full(len(y), base_rate))
            out["brier_skill"] = 1 - out["brier"] / ref
    return out


def run_logit(df, folds, fs, freq, label, subsample="all"):
    feats = FEATURE_SETS[fs]
    X, y = df[feats].to_numpy(float), df[label].to_numpy()
    rows, preds, coefs = [], [], []
    for k, (tr, va, te) in enumerate(folds, start=1):
        fit = fit_slice(df, tr, va)
        best = (np.inf, None)
        for c in LOGIT_C_GRID:
            p = logit_factory(feats, c).fit(X[fit], y[fit]).predict_proba(X[va])[:, 1]
            ll = log_loss(y[va], np.clip(p, EPS, 1 - EPS), labels=[0, 1])
            if ll < best[0]:
                best = (ll, c)
        est = logit_factory(feats, best[1]).fit(X[tr], y[tr])
        p = est.predict_proba(X[te])[:, 1]
        m = logit_scores(y[te], p, y[tr].mean())
        rows.append({"model": LOGIT_NAME, "feature_set": fs, "freq": freq,
                     "label": label, "fold": k,
                     "cutoff_date": folds.cutoffs[k - 1].date().isoformat(),
                     "n_train": len(tr), "n_test": len(te),
                     "best_params": f'{{"C": {best[1]}}}', **m,
                     "subsample": subsample})
        preds.append(pd.DataFrame({"token_id": df["token_id"].to_numpy()[te],
                                   "date": df["date"].to_numpy()[te],
                                   "y_true": y[te], "p": p, "fold": k}))
        for f, c in zip(feats, est.named_steps["logit"].coef_[0]):
            coefs.append({"label": label, "feature_set": fs, "freq": freq,
                          "fold": k, "feature": f, "coef_std": c,
                          "odds_ratio_per_sd": np.exp(c)})
        print(f"[{LOGIT_NAME}|{label}|{fs}_{freq}] fold {k}: val logloss {best[0]:.4f} "
              f"test auc {m['auc']:.4f} brier_skill {m['brier_skill']:+.4f} "
              f"C={best[1]:g}")
    if not SMOKE:
        pd.concat(preds, ignore_index=True).to_parquet(
            RESULTS_DIR / f"preds_{LOGIT_NAME}_{label}_{run_tag(fs, freq, subsample=subsample)}.parquet",
            index=False)
    return pd.DataFrame(rows), pd.DataFrame(coefs)


def naive_rows(df, folds, freq, label, subsample="all"):
    """Rank by rv_past_14d: AUC / PR-AUC only (it is a score, not a probability).
    Only meaningful for top20_xs; spike_1.5x is relative to the token's own vol."""
    rows = []
    for k, (tr, _, te) in enumerate(folds, start=1):
        m = logit_scores(df[label].to_numpy()[te], df["rv_past_14d"].to_numpy()[te],
                         prob=False)
        rows.append({"model": "naive_rv14", "feature_set": "NONE", "freq": freq,
                     "label": label, "fold": k,
                     "cutoff_date": folds.cutoffs[k - 1].date().isoformat(),
                     "n_train": len(tr), "n_test": len(te), "best_params": "{}",
                     **m, "subsample": subsample})
    return pd.DataFrame(rows)


def upsert_logit(path: Path, new: pd.DataFrame) -> None:
    if path.exists():
        old = pd.read_csv(path)
        keys = set(map(tuple, new[LOGIT_KEY].astype(str).drop_duplicates().to_numpy()))
        keep = [tuple(r) not in keys for r in old[LOGIT_KEY].astype(str).to_numpy()]
        new = pd.concat([old[keep], new], ignore_index=True)
    new[LOGIT_COLUMNS].to_csv(path, index=False)

In [ ]:
# about 1.5 minutes in total, so it is always recomputed
daily_l = add_labels(daily)
weekly_l = weekly.merge(daily_l[["token_id", "date"] + LABELS],
                        on=["token_id", "date"], how="left", validate="one_to_one")
assert weekly_l[LABELS].notna().all().all()
print("base rates (daily): " + ", ".join(f"{l}={daily_l[l].mean():.3f}" for l in LABELS))

logit_rows, logit_coef_frames = [], []
for freq, df_l in (("daily", daily_l), ("weekly", weekly_l)):
    folds_l = data[freq][1]          # built from the same rows in the same order
    for label in LABELS:
        if label == "top20_xs":
            logit_rows.append(naive_rows(df_l, folds_l, freq, label))
        for fs in ("CONTROLS", "FULL"):
            r, c = run_logit(df_l, folds_l, fs, freq, label)
            logit_rows.append(r)
            logit_coef_frames.append(c)
sub_l = daily_l[daily_l["listedMethod_INTERNAL"] == 1].reset_index(drop=True)
sub_l_folds = make_folds(sub_l, verbose=False)
for label in LABELS:
    if label == "top20_xs":
        logit_rows.append(naive_rows(sub_l, sub_l_folds, "daily", label, "INTERNAL"))
    for fs in ("CONTROLS", "FULL"):
        logit_rows.append(run_logit(sub_l, sub_l_folds, fs, "daily", label, "INTERNAL")[0])

logit_res = pd.concat(logit_rows, ignore_index=True)
logit_coefs = pd.concat(logit_coef_frames, ignore_index=True)
if not SMOKE:
    upsert_logit(RESULTS_DIR / f"{LOGIT_NAME}.csv", logit_res)
    logit_coefs.to_csv(RESULTS_DIR / "logistic_coefs.csv", index=False)

print("Logistic mean over folds:")
display(logit_res.groupby(["label", "freq", "subsample", "model", "feature_set"],
                          sort=False)[["auc", "pr_auc", "brier", "log_loss",
                                       "brier_skill"]].mean().round(4))

In [ ]:
def plot_auc(res):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    sets = [("CONTROLS", "controls only", PALETTE[0]),
            ("FULL", "controls + emission + static (FULL)", PALETTE[1])]
    for ax, freq in zip(axes, ("daily", "weekly")):
        r = res[(res.freq == freq) & (res.subsample == "all")]
        x = np.arange(len(LABELS))
        for i, (fs, lab, color) in enumerate(sets):
            g = r[r.feature_set == fs].groupby("label")["auc"]
            mean, std = g.mean().reindex(LABELS), g.std().reindex(LABELS)
            ax.bar(x + (i - 0.5) * 0.38, mean, width=0.36, yerr=std, color=color,
                   label=lab, error_kw={"ecolor": INK_2, "elinewidth": 1})
            for xi, v in zip(x, mean):
                ax.text(xi + (i - 0.5) * 0.38, 0.505, f"{v:.3f}", ha="center",
                        va="bottom", fontsize=8, color="white", fontweight="bold")
        naive = r[r.model == "naive_rv14"].groupby("label")["auc"].mean()
        for xi, l in enumerate(LABELS):
            if l not in naive:
                continue
            ax.hlines(naive[l], xi - 0.4, xi + 0.4, color=INK, linestyle="--",
                      linewidth=1.2, label="rank by rv_past_14d" if xi == 0 else None)
        ax.axhline(0.5, color=INK_2, linestyle=":", linewidth=1)
        ax.set_ylim(0.5, None)
        ax.set_xticks(x)
        ax.set_xticklabels(LABELS)
        ax.grid(axis="x", visible=False)
        ax.set_title(f"{freq} panel", fontsize=10)
    axes[0].set_ylabel("test ROC-AUC (mean of 5 folds, bar = +/- 1 std)")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.08))
    fig.suptitle("Logistic regression: high-volatility events, controls only vs "
                 "FULL", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    savefig(fig, "logistic_auc.png")


plot_auc(logit_res)
m = logit_res[(logit_res.model == LOGIT_NAME) & (logit_res.subsample == "all")]
ll = m.pivot_table(index=["label", "freq"], columns="feature_set", values="log_loss")
ll["delta_FULL_minus_CONTROLS"] = ll["FULL"] - ll["CONTROLS"]
print("Test log loss (mean of 5 folds); positive delta = FULL is worse")
display(ll.round(4))

## 9. Comparison across models

Every per-fold results CSV in `results/` is aggregated into one table (mean and std across folds by model, feature set, panel frequency, target and subsample). The chart shows `r2_oos` against persistence for controls-only vs the full feature set.


In [ ]:
METRICS = ["rmse", "mae", "r2_oos", "spearman", "rmse_level"]
REQUIRED = set(RESULT_COLUMNS) - {"subsample"}
BASELINES = ["persistence", "train_mean"]


def read_results() -> pd.DataFrame:
    frames = []
    for path in sorted(RESULTS_DIR.glob("*.csv")):
        head = pd.read_csv(path, nrows=0)
        if not REQUIRED.issubset(head.columns):
            continue  # eda summary, importances, validation curves, ...
        d = pd.read_csv(path)
        if "subsample" not in d.columns:
            d["subsample"] = "all"
        d["subsample"] = d["subsample"].fillna("all")
        frames.append(d)
        print(f"[04_compare] {path.name}: {len(d)} fold rows, "
              f"models {sorted(d['model'].unique())}")
    return pd.concat(frames, ignore_index=True)

def plot(table):
    main_t = table[(table.target == TARGETS["y"]) & (table.subsample == "all")]
    models = [m for m in main_t["model"].unique() if m not in BASELINES]
    sets = [("CONTROLS", "controls only", PALETTE[0]),
            ("FULL", "controls + emission + static (FULL)", PALETTE[1])]
    fig, axes = plt.subplots(1, 2, figsize=(max(9, 2.2 * len(models) + 5), 5),
                             sharey=True)
    for ax, freq in zip(axes, ("daily", "weekly")):
        t = main_t[main_t.freq == freq].set_index(["model", "feature_set"])
        x = np.arange(len(models))
        for i, (fs, label, color) in enumerate(sets):
            mean = [t["r2_oos_mean"].get((m, fs), np.nan) for m in models]
            std = [t["r2_oos_std"].get((m, fs), np.nan) for m in models]
            ax.bar(x + (i - 0.5) * 0.38, mean, width=0.36, yerr=std, color=color,
                   label=label, error_kw={"ecolor": INK_2, "elinewidth": 1})
            for xi, v in zip(x, mean):
                if np.isfinite(v):
                    ax.text(xi + (i - 0.5) * 0.38, 0.005, f"{v:.3f}", ha="center",
                            va="bottom", fontsize=8, color="white",
                            fontweight="bold")
        ax.axhline(0, color=INK, linestyle="--", linewidth=1.2,
                   label="persistence baseline (0 by construction)")
        if ("train_mean", "NONE") in t.index:
            tm = t.loc[("train_mean", "NONE"), "r2_oos_mean"]
            ax.axhline(tm, color=INK_2, linestyle=":", linewidth=1.5,
                       label="pooled train-mean baseline")
            ax.text(-0.58, tm, f"{tm:.3f}", ha="left", va="bottom", fontsize=8,
                    color=INK_2)
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.set_xlim(-0.6, len(models) - 0.4)
        ax.grid(axis="x", visible=False)
        ax.set_title(f"{freq} panel", fontsize=10)
    axes[0].set_ylabel("r2_oos vs persistence (mean of 5 folds, bar = +/- 1 std)")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2,
               bbox_to_anchor=(0.5, -0.1))
    fig.suptitle("Out-of-sample R2 by model: controls only vs full feature set "
                 "(target log rv_fwd_14d)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    savefig(fig, "compare_r2_oos.png")


In [ ]:
if SMOKE:
    res_all = pd.concat([baseline_res, knn_res, rf_res, linear_res], ignore_index=True)
else:
    res_all = read_results()          # every per-fold results CSV in results/
keys = ["model", "feature_set", "freq", "target", "subsample"]
table = res_all.groupby(keys, sort=False)[METRICS].agg(["mean", "std"])
table.columns = [f"{m}_{s}" for m, s in table.columns]
table.insert(0, "n_folds", res_all.groupby(keys, sort=False)["fold"].nunique())
table = table.reset_index().sort_values(
    ["target", "subsample", "freq", "model", "feature_set"],
    key=lambda s: s.map({"daily": 0, "weekly": 1}) if s.name == "freq" else s)
if not SMOKE:
    table.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)
with pd.option_context("display.max_rows", 200):
    display(table.round(4))
plot(table)


In [ ]:
main_t = table[(table.target == TARGETS["y"]) & (table.subsample == "all")]
piv = main_t.pivot(index=["freq", "model"], columns="feature_set", values="r2_oos_mean")
piv["delta_FULL_minus_CONTROLS"] = piv["FULL"] - piv["CONTROLS"]
print("Headline: out-of-sample R2 vs persistence (mean of 5 folds), target log rv_fwd_14d")
display(piv.round(4))
rmse = main_t.pivot(index=["freq", "model"], columns="feature_set", values="rmse_mean")
rmse["delta_FULL_minus_CONTROLS"] = rmse["FULL"] - rmse["CONTROLS"]
print("RMSE in log-vol space (lower is better)")
display(rmse.round(4))


## 10. Conclusions

*(filled in after the full run; see the headline table above)*
